# Replicant — Live Metrics (PromQL)

Queries a running Prometheus instance for replica-side metrics. Unlike [`convergence.ipynb`](convergence.ipynb) (CSV) and [`protocol_metrics.ipynb`](protocol_metrics.ipynb) (OTel JSON files), this notebook has no offline data source — it only produces output when a stack is running.

**Bring up a stack first**, then run a scenario against it:

```sh
# Docker:
just docker-up scenarios/full-mesh-n5.toml
cargo run --release --bin orchestrator -- \
  --replicas localhost:50051=replica-0:50051,localhost:50052=replica-1:50051,\
localhost:50053=replica-2:50051,localhost:50054=replica-3:50051,localhost:50055=replica-4:50051 \
  scenarios/full-mesh-n5.toml

# Or k8s (port-forwards in the foreground):
just k8s-up scenarios/full-mesh-n5.toml
just k8s-ui  # Grafana :3000, Prometheus :9090
```

Prometheus is expected at `http://localhost:9090`. If it's unreachable the cells below skip themselves rather than erroring.

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 150

## Prometheus-backed metrics (live stack)

This section pulls metrics from a running Prometheus instance instead of the per-scenario JSON files above. The intended flow:

```sh
# 1. Bring up a sized stack and leave it running. `docker-up` generates a
# compose file from the scenario's node_count, builds the image, and starts
# the replicas + otel-collector + prometheus.
just docker-up scenarios/full-mesh-n5.toml

# 2. Run a scenario against the running stack. The --replicas flag maps
# host ports (orchestrator dials these) to container DNS names (replicas
# dial these for peer connections). For n=5:
cargo run --release --bin orchestrator -- \
  --replicas localhost:50051=replica-0:50051,localhost:50052=replica-1:50051,\
localhost:50053=replica-2:50051,localhost:50054=replica-3:50051,localhost:50055=replica-4:50051 \
  scenarios/full-mesh-n5.toml

# 3. Inspect Prometheus at http://localhost:9090, then re-run the cells below.

# 4. Tear down when done.
just docker-down
```

If Prometheus isn't reachable (no stack up), this section is skipped — the JSON-file sections above remain the primary path for offline thesis runs.

In [ ]:
import requests

PROM = "http://localhost:9090"


def prom_query(q: str, prom: str = PROM) -> list[dict]:
    """Run a PromQL instant query. Returns [] if Prometheus is unreachable."""
    try:
        r = requests.get(f"{prom}/api/v1/query", params={"query": q}, timeout=2)
        r.raise_for_status()
        data = r.json()
        return data["data"]["result"] if data.get("status") == "success" else []
    except (requests.RequestException, ValueError):
        return []


prom_alive = bool(prom_query("up"))
print(f"Prometheus at {PROM}: {'reachable' if prom_alive else 'unreachable — section skipped'}")

### Sync message traffic per actor

Total sent (`tx`) and received (`rx`) sync messages per actor, aggregated across all peers. Same view as the JSON-file section above, but live from the TSDB.

In [ ]:
if prom_alive:
    def _to_df(result, value_col):
        return pd.DataFrame(
            [{"actor": s["metric"]["actor"], value_col: int(float(s["value"][1]))} for s in result]
        )

    tx = _to_df(prom_query("sum by (actor) (replicant_sync_messages_tx_total)"), "tx")
    rx = _to_df(prom_query("sum by (actor) (replicant_sync_messages_rx_total)"), "rx")
    traffic = (
        tx.set_index("actor")
          .join(rx.set_index("actor"), how="outer")
          .fillna(0).astype(int)
    )
    traffic = traffic.loc[sorted(traffic.index, key=lambda s: int(s.split("-")[1]))]

    fig, ax = plt.subplots(figsize=(max(4, len(traffic) * 0.9), 4))
    x = range(len(traffic))
    width = 0.35
    ax.bar([i - width / 2 for i in x], traffic["tx"], width, label="sent (tx)", alpha=0.8)
    ax.bar([i + width / 2 for i in x], traffic["rx"], width, label="received (rx)", alpha=0.8)
    ax.set_xticks(list(x))
    ax.set_xticklabels(traffic.index)
    ax.set_xlabel("Actor")
    ax.set_ylabel("Sync messages")
    ax.set_title("Sync message traffic per actor (from Prometheus)")
    ax.legend()
    fig.tight_layout()
    plt.show()
    print(traffic.assign(total=traffic.tx + traffic.rx))

### Document size — post-convergence (live stack)

Live equivalent of the per-scenario table above. After every `sync_receive` the replica re-samples `replicant.doc.size_bytes`, so each actor's latest gauge value reflects post-convergence state.

The CRDT convergence invariant (fingerprint equality) is enforced by the orchestrator; this cell surfaces the `save()` byte spread as a property of the encoding rather than asserting strict equality. For full-mesh scenarios — the only thing `just smoke-docker` runs today — spread is expected to be 0; if you point this at a non-mesh scenario you should expect a few-percent spread, per the analysis above.

In [ ]:
if prom_alive:
    doc = (
        pd.DataFrame(
            [
                {"actor": s["metric"]["actor"], "bytes": int(float(s["value"][1]))}
                for s in prom_query("replicant_doc_size_bytes")
            ]
        )
        .sort_values("actor", key=lambda s: s.map(lambda v: int(v.split("-")[1])))
        .reset_index(drop=True)
    )

    fig, ax = plt.subplots(figsize=(max(4, len(doc) * 0.9), 4))
    ax.bar(doc["actor"], doc["bytes"], alpha=0.8)
    ax.set_xlabel("Actor")
    ax.set_ylabel("Document size (bytes)")
    ax.set_title("Post-convergence document size per actor (from Prometheus)")
    fig.tight_layout()
    plt.show()

    spread = int(doc["bytes"].max() - doc["bytes"].min())
    spread_pct = (spread / int(doc["bytes"].min()) * 100) if len(doc) else 0.0
    print(doc.to_string(index=False))
    print(f"\nspread: {spread} bytes ({spread_pct:.1f}%)")
    # Soft check matching the per-scenario cell above. A few-percent spread
    # is the Automerge save()-ordering effect; anything much larger is a
    # real sync/timing bug.
    assert spread_pct <= 10.0, (
        f"doc_size spread {spread_pct:.1f}% exceeds 10% — investigate"
    )
    if spread == 0:
        print("✓ save() bytes identical across replicas (expected for full-mesh)")
    else:
        print(f"✓ save() bytes within expected encoding-ordering tolerance")